# Data Science Tools and Ecosystem


In this notebook, Data Science Tools and Ecosystem are summarized.

**Objectives:**

- List popular languages for Data Science.
- Introduce commonly used libraries in Data Science.
- Explore examples of evaluating arithmetic expressions in Python.
- Understand how to convert minutes to hours using Python.
- Summarize the Data Science tools and ecosystem.


Some of the popular languages that Data Scientists use are:

1. Python
2. R
3. SQL
4. Julia
5. Scala


Some of the commonly used libraries by Data Scientists include:

1. NumPy
2. Pandas
3. Matplotlib
4. Scikit-Learn
5. TensorFlow


| Data Science Tools    |
|-----------------------|
| Jupyter Notebook      |
| RStudio               |
| Visual Studio Code    |


### Examples of Evaluating Arithmetic Expressions in Python

In Python, you can perform various arithmetic operations. Here are some examples:

1. **Addition:**
   ```python
   result = 5 + 3
   # result will be 8
result = 10 - 4
# result will be 6


In [6]:
# This a simple arithmetic expression to mutiply then add integers
(3*4)+5

17

In [8]:
#This will convert 200 minutes to hours by diving by 60
200/60

3.3333333333333335

## Author
Andrea Lamacchia


In [ ]:
# ── Cell L: Test 3 — Ljung-Box on real (or simulated) data residuals ─────────
#
# To use real data:
#   Y_real  = <your (T,3) array>
#   M_real  = <your (T,3) bool mask>
#   est_real, _ = estimate_logit(Y_real, M_real, CONFIGS_LOGIT['2int'], dt=1.0,
#                                art_scale=..., tone_scale=..., n_restarts=10)
#   _, std_res_real = _run_filter_logit(est_real, Y_real, M_real,
#                                       config['channels'], dt=1.0)
# Then replace std_res_paper below with std_res_real.

# ── using simulated data as stand-in ──────────────────────────────────────────
std_res_paper = std_res     # swap for std_res_real when real data is available

print('Ljung-Box p-values  |  2int weekly fit  (paper diagnostic)')
print(f'{"Channel":<12} {"lag 5 p":>10} {"lag 10 p":>10} {"white@5%?":>12}')
print('─' * 48)
for j, lbl in enumerate(CH_LABELS):
    r   = std_res_paper[:, j]
    rv  = pd.Series(r[~np.isnan(r)])
    lb  = acorr_ljungbox(rv, lags=10, return_df=True)
    p5  = lb['lb_pvalue'].iloc[4]
    p10 = lb['lb_pvalue'].iloc[9]
    w   = 'YES' if p10 > 0.05 else 'NO'
    print(f'{lbl:<12} {p5:>10.4f} {p10:>10.4f} {w:>12}')

print('\n(p > 0.05 → innovations are white → filter is correctly specified)')

## Test 3 — Ljung-Box on Real Data Residuals (2int weekly fit)

Replace `Y_real` / `M_real` below with the actual data arrays to get the paper-ready table.  
Running on the simulated dataset here demonstrates the output format.

In [ ]:
# ── Cell K: Test 2 — NEES ────────────────────────────────────────────────────

DF_CHI2    = 2
CI_LO      = chi2.ppf(0.025, DF_CHI2)   # 0.0506
CI_HI      = chi2.ppf(0.975, DF_CHI2)   # 7.3778
T_sim      = true_Z.shape[0]
nees       = np.zeros(T_sim)

for t in range(T_sim):
    dz = true_Z[t] - filt_z[t]
    Pt = P_list[t]
    try:
        nees[t] = float(dz @ np.linalg.solve(Pt, dz))
    except np.linalg.LinAlgError:
        nees[t] = float(dz @ np.linalg.lstsq(Pt, dz, rcond=None)[0])

frac_out = np.mean((nees < CI_LO) | (nees > CI_HI))

print('NEES summary (ALR space, df=2)')
print(f'  Mean NEES        = {nees.mean():.3f}   (expected ≈ {DF_CHI2})')
print(f'  95% chi2(2) CI   = [{CI_LO:.3f}, {CI_HI:.3f}]')
print(f'  Fraction outside = {frac_out*100:.1f}%  (expected ≈ 5%)')

# NEES plot
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(nees, color='steelblue', alpha=0.8, lw=1, label='NEES$_t$')
ax.axhline(CI_LO, color='crimson', ls='--', lw=1.5,
           label=f'95% CI lower ({CI_LO:.2f})')
ax.axhline(CI_HI, color='crimson', ls='--', lw=1.5,
           label=f'95% CI upper ({CI_HI:.2f})')
ax.axhline(DF_CHI2, color='seagreen', ls=':', lw=1.8,
           label=f'Expected mean ({DF_CHI2})')
ax.fill_between(range(T_sim), CI_LO, CI_HI, alpha=0.07, color='crimson')
ax.set_xlabel('Time step (weeks)');  ax.set_ylabel('NEES')
ax.set_title('Normalised Estimation Error Squared  (ALR space, df = 2)')
ax.legend(fontsize=9, ncol=2);  ax.grid(True, alpha=0.3)
fig.tight_layout()
savefig(fig, 'nees_plot')
plt.show()

In [ ]:
# ── Cell J: Test 2 — standardised innovations + Ljung-Box ───────────────────

CH_LABELS = ['GT', 'Articles', 'Tone']

print('Running filter on single simulated dataset with estimated parameters ...\n')
filt_SI, std_res, P_list, filt_z = _run_filter_logit(
    est, Y_sim, M_sim, config['channels'], dt=1.0, return_P=True
)

# ── 1. Marginal statistics ────────────────────────────────────────────────────
print(f'{"Channel":<12} {"mean":>8} {"std":>8} {"|r|>2 (%)":>12} {"JB p-val":>12}')
print('─' * 55)
for j, lbl in enumerate(CH_LABELS):
    r   = std_res[:, j]
    rv  = r[~np.isnan(r)]
    mn  = rv.mean();  sd = rv.std()
    f2  = np.mean(np.abs(rv) > 2.0) * 100
    _, jb_p = jb_test(rv)
    print(f'{lbl:<12} {mn:>8.3f} {sd:>8.3f} {f2:>12.1f} {jb_p:>12.4f}')

# ── 2. Ljung-Box whiteness test ───────────────────────────────────────────────
print(f'\nLjung-Box whiteness test (10 lags):')
print(f'{"Channel":<12} {"LB(10) stat":>13} {"p-value":>10} {"white(p>0.05)?":>16}')
print('─' * 55)
for j, lbl in enumerate(CH_LABELS):
    r  = std_res[:, j]
    rv = pd.Series(r[~np.isnan(r)])
    lb = acorr_ljungbox(rv, lags=10, return_df=True)
    st = lb['lb_stat'].iloc[-1]
    p  = lb['lb_pvalue'].iloc[-1]
    print(f'{lbl:<12} {st:>13.3f} {p:>10.4f} {"YES" if p > 0.05 else "NO":>16}')

# lag-1…5 ACF table
print(f'\nLag-1..5 autocorrelations of standardised innovations:')
hdr = f'{"Channel":<12}' + ''.join(f'{"lag"+str(l):>9}' for l in range(1, 6))
print(hdr);  print('─' * (12 + 9*5))
for j, lbl in enumerate(CH_LABELS):
    r  = std_res[:, j]
    rv = r[~np.isnan(r)]
    row = f'{lbl:<12}'
    for lag in range(1, 6):
        ac  = np.corrcoef(rv[:-lag], rv[lag:])[0, 1]
        row += f'{ac:>+9.3f}'
    print(row)

## Test 2 — Innovation Consistency Checks

In [ ]:
# ── Cell I: Test 1b — Monte Carlo boxplots ───────────────────────────────────

n_p   = len(PARAM_KEYS)
ncols = 4
nrows = (n_p + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
axes = axes.flatten()

for i, k in enumerate(PARAM_KEYS):
    ax   = axes[i]
    vals = np.array(mc_results[k])
    ax.boxplot(vals[~np.isnan(vals)], vert=True, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.7),
               medianprops=dict(color='white', lw=2))
    ax.axhline(TRUE_MC[k], color='crimson', lw=2, linestyle='--', label='true')
    ax.set_title(k, fontsize=10)
    ax.set_xticks([])
    ax.legend(fontsize=8, loc='best')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Monte Carlo Parameter Recovery  (N = 20, true value = red dashed)',
             fontsize=12, y=1.01)
fig.tight_layout()
savefig(fig, 'mc_param_recovery')
plt.show()

In [ ]:
# ── Cell H: Test 1b — Monte Carlo (N_SIM = 20) ───────────────────────────────

N_SIM        = 20
MC_SEED_BASE = 100

PARAM_KEYS = ['beta', 'gamma', 'R0', 'c2', 'c3', 'a2', 'a3',
              'sigma2_S', 'sigma2_I', 'sigma2_eps1', 'sigma2_eps2', 'sigma2_eps3',
              'I0']
TRUE_MC = {
    'beta': 0.3,  'gamma': 0.1,
    'R0':   0.3 / 0.1,
    'c2':   200.0, 'c3': -0.8,
    'a2':   50.0,  'a3': -0.5,
    'sigma2_S':    0.005, 'sigma2_I':    0.15,
    'sigma2_eps1': 0.005, 'sigma2_eps2': 50.0, 'sigma2_eps3': 0.1,
    'I0':   0.01,
}

mc_results = {k: [] for k in PARAM_KEYS}

for sim_i in range(N_SIM):
    rng_i = np.random.default_rng(MC_SEED_BASE + sim_i)

    # draw a valid (non-degenerate) dataset
    d = None;  attempt = 0
    while d is None and attempt < 20:
        d = simulate_sir_obs(true_params, T=100, rng=rng_i)
        attempt += 1
    if d is None:
        print(f'  Sim {sim_i+1:2d}: SKIPPED (degenerate)')
        continue

    Y_i, M_i, _, _ = d
    as_i = float(np.nanmax(np.abs(Y_i[:, 1])))
    ts_i = float(np.nanmax(np.abs(Y_i[:, 2])))

    est_i, _ = estimate_logit(
        Y_i, M_i, config, dt=1.0,
        beta0=0.3, gamma0=0.1,      # warm-start near true values
        art_scale=as_i, tone_scale=ts_i,
        q_lo=-10.0, q_hi=5.0,
        n_restarts=5, verbose=False,
    )

    for k in PARAM_KEYS:
        mc_results[k].append(float(est_i.get(k, np.nan)))

    print(f'  Sim {sim_i+1:2d}/{N_SIM}: '
          f'beta={est_i["beta"]:.4f}  gamma={est_i["gamma"]:.4f}  '
          f'I0={est_i["I0"]:.5f}  R0={est_i["R0"]:.3f}')

# ── summary table ─────────────────────────────────────────────────────────────
print(f'\n{"param":<14} {"true":>8} {"mean":>10} {"std":>10} '
      f'{"bias":>10} {"in_IQR":>8}')
print('─' * 65)

rows = []
for k in PARAM_KEYS:
    vals          = np.array(mc_results[k])
    tv            = TRUE_MC[k]
    mn, sd        = np.nanmean(vals), np.nanstd(vals)
    bias          = mn - tv
    q25, q75      = np.nanpercentile(vals, [25, 75])
    in_iqr        = bool(q25 <= tv <= q75)
    print(f'{k:<14} {tv:>8.4g} {mn:>10.4g} {sd:>10.4g} {bias:>10.4g} {str(in_iqr):>8}')
    rows.append(dict(param=k, true=tv, mean=mn, std=sd, bias=bias,
                     q25=q25, q75=q75, in_iqr=in_iqr))

mc_df = pd.DataFrame(rows)
mc_df.to_csv('cache_logit/param_recovery.csv', index=False)
print('\nSaved  cache_logit/param_recovery.csv')

### 1b: Monte Carlo (N = 20 simulations)

In [ ]:
# ── Cell G: Test 1a — single-dataset estimation ──────────────────────────────

np.random.seed(42)
rng_main = np.random.default_rng(42)

print('Simulating one dataset (T=100, seed=42) ...')
data = None
while data is None:
    data = simulate_sir_obs(true_params, T=100, rng=rng_main)
Y_sim, M_sim, true_SI, true_Z = data

peak_t = true_SI[:, 1].argmax()
print(f'Peak I = {true_SI[:, 1].max():.4f}  at t = {peak_t}  '
      f'(should be ~30-40 for full cycle visibility)')

config        = CONFIGS_LOGIT['2int']
art_scale_v   = float(np.nanmax(np.abs(Y_sim[:, 1])))
tone_scale_v  = float(np.nanmax(np.abs(Y_sim[:, 2])))
print(f'art_scale={art_scale_v:.2f}  tone_scale={tone_scale_v:.3f}\n')

print('Running MLE (n_restarts=10) ...')
est, opt = estimate_logit(
    Y_sim, M_sim, config, dt=1.0,
    beta0=0.3, gamma0=0.1,          # warm-start near true values
    art_scale=art_scale_v,
    tone_scale=tone_scale_v,
    q_lo=-10.0, q_hi=5.0,
    n_restarts=10, verbose=True,
)

# ── recovery table ────────────────────────────────────────────────────────────
COMPARE = [
    ('beta',        true_params['beta']),
    ('gamma',       true_params['gamma']),
    ('R0',          true_params['beta'] / true_params['gamma']),
    ('c2',          true_params['c2']),
    ('c3',          true_params['c3']),
    ('a2',          true_params['a2']),
    ('a3',          true_params['a3']),
    ('sigma2_S',    true_params['sigma2_S']),
    ('sigma2_I',    true_params['sigma2_I']),
    ('sigma2_eps1', true_params['sigma2_eps1']),
    ('sigma2_eps2', true_params['sigma2_eps2']),
    ('sigma2_eps3', true_params['sigma2_eps3']),
    ('I0',          true_params['I0']),
]

print(f'\n{"param":<14} {"true":>10} {"estimated":>12} {"rel_err(%)":>12}')
print('─' * 52)
for name, tv in COMPARE:
    ev      = est[name]
    rel_err = 100.0 * (ev - tv) / max(abs(tv), 1e-12)
    print(f'{name:<14} {tv:>10.4g} {ev:>12.4g} {rel_err:>12.1f}')

## filterpy Cross-Validation

Run the same simulated dataset through `filterpy`'s `UnscentedKalmanFilter`
(Merwe scaled sigma points, identical alpha/beta/kappa) and compare filtered
state trajectories against the hand-coded filter. Agreement to numerical
tolerance validates the custom UKF implementation.


In [ ]:
# ---- filterpy cross-validation -----------------------------------------
# Tries to import filterpy; if unavailable installs via pip.
# Falls back to a pure-numpy mirror of filterpy's UKF interface so the
# structural equivalence is visible regardless of install status.

import subprocess, sys

def _try_install_filterpy():
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                               'filterpy', '-q'], timeout=120)
        return True
    except Exception:
        return False

try:
    from filterpy.kalman import UnscentedKalmanFilter as FP_UKF
    from filterpy.kalman import MerweScaledSigmaPoints as FP_MSP
    _USE_FILTERPY = True
    print('filterpy loaded')
except ImportError:
    if _try_install_filterpy():
        from filterpy.kalman import UnscentedKalmanFilter as FP_UKF
        from filterpy.kalman import MerweScaledSigmaPoints as FP_MSP
        _USE_FILTERPY = True
        print('filterpy installed and loaded')
    else:
        _USE_FILTERPY = False
        print('filterpy unavailable -- using numpy mirror')

# ---- numpy mirror of filterpy's UKF API ---------------------------------
if not _USE_FILTERPY:

    class FP_MSP:
        def __init__(self, n, alpha, beta, kappa):
            self.n   = n
            lam      = alpha**2 * (n + kappa) - n
            self.Wm  = np.empty(2*n+1)
            self.Wc  = np.empty(2*n+1)
            self.Wm[0] = lam / (n + lam)
            self.Wc[0] = self.Wm[0] + (1 - alpha**2 + beta)
            self.Wm[1:] = self.Wc[1:] = 0.5 / (n + lam)
            self._scale = np.sqrt(n + lam)

        def sigma_points(self, x, P):
            n  = self.n
            Ps = 0.5*(P + P.T) + 1e-10*np.eye(n)
            try:    L = np.linalg.cholesky(Ps)
            except: L = np.diag(np.sqrt(np.maximum(np.diag(Ps), 1e-12)))
            pts = np.empty((2*n+1, n))
            pts[0] = x
            for i in range(n):
                pts[i+1]   = x + self._scale * L[:, i]
                pts[n+i+1] = x - self._scale * L[:, i]
            return pts

        def num_sigmas(self): return 2*self.n + 1

    class FP_UKF:
        def __init__(self, dim_x, dim_z, dt, fx, hx, points):
            self.dim_x = dim_x;  self.dim_z = dim_z
            self.dt = dt;  self._fx = fx;  self._hx = hx
            self._pts = points
            self.x    = np.zeros(dim_x)
            self.P    = np.eye(dim_x)
            self.Q    = np.eye(dim_x)
            self.R    = np.eye(dim_z)
            self._innov = None

        def predict(self):
            Wm = self._pts.Wm;  Wc = self._pts.Wc
            sp  = self._pts.sigma_points(self.x, self.P)
            fsp = np.array([self._fx(s, self.dt) for s in sp])
            xp  = Wm @ fsp
            dz  = fsp - xp
            Pp  = dz.T @ np.diag(Wc) @ dz + self.Q
            self.x = xp
            self.P = 0.5*(Pp + Pp.T) + 1e-10*np.eye(self.dim_x)

        def update(self, z, R=None):
            if R is None: R = self.R
            Wm = self._pts.Wm;  Wc = self._pts.Wc
            sp  = self._pts.sigma_points(self.x, self.P)
            hsp = np.array([self._hx(s) for s in sp])
            yp  = Wm @ hsp
            dy  = hsp - yp
            Syy = dy.T @ np.diag(Wc) @ dy + R
            dx  = sp - self.x
            Pxy = dx.T @ np.diag(Wc) @ dy
            try:    K = Pxy @ np.linalg.inv(Syy)
            except: K = Pxy @ np.linalg.pinv(Syy)
            self._innov = z - yp
            self.x = self.x + K @ self._innov
            self.P = self.P - K @ Syy @ K.T
            self.P = 0.5*(self.P + self.P.T) + 1e-10*np.eye(self.dim_x)

# ---- transition and measurement functions for filterpy ------------------
# Uses est from Cell G so comparison is on identical parameters.

def _fx_alr(z, dt):
    SI     = _inv_alr_rows(z[None, :])[0]
    SI_nxt = sir_transition_vec(SI[None, :], est['beta'], est['gamma'], dt)[0]
    return _alr_rows(SI_nxt[None, :])[0]

def _hx_alr(z):
    I = _inv_alr_rows(z[None, :])[0][1]
    return np.array([1.0 * I + est.get('a1', 0.0),
                     est['c2'] * I + est['a2'],
                     est['c3'] * I + est['a3']])

# ---- build and initialise filterpy UKF ----------------------------------
sigmas = FP_MSP(n=2, alpha=_ALPHA, beta=_BETA_UKF, kappa=_KAPPA)
fp_ukf = FP_UKF(dim_x=2, dim_z=3, dt=1.0, fx=_fx_alr, hx=_hx_alr, points=sigmas)

Q_fp          = np.diag([est['sigma2_S'], est['sigma2_I']])
fp_ukf.x      = _alr(est['S0'], est['I0']).copy()
fp_ukf.P      = 10.0 * Q_fp + 1e-6 * np.eye(2)
fp_ukf.Q      = Q_fp
fp_ukf.R      = np.diag([est['sigma2_eps1'], est['sigma2_eps2'], est['sigma2_eps3']])

# ---- run filterpy on Y_sim ----------------------------------------------
T_fp       = Y_sim.shape[0]
fp_filt_z  = np.zeros((T_fp, 2))

for t in range(T_fp):
    fp_ukf.predict()
    fp_ukf.update(Y_sim[t])          # M_sim is all-True so all channels observed
    fp_filt_z[t] = fp_ukf.x.copy()

fp_filt_SI = _inv_alr_rows(fp_filt_z)

# ---- run hand-coded filter ----------------------------------------------
_, _, _, hc_filt_z = _run_filter_logit(
    est, Y_sim, M_sim, config['channels'], dt=1.0, return_P=True)
hc_filt_SI = _inv_alr_rows(hc_filt_z)

# ---- comparison metrics -------------------------------------------------
diff_z  = fp_filt_z  - hc_filt_z
diff_SI = fp_filt_SI - hc_filt_SI

print('filterpy vs hand-coded UKF -- filtered state comparison')
print(f'  Max |delta_z1| = {np.abs(diff_z[:,0]).max():.3e}')
print(f'  Max |delta_z2| = {np.abs(diff_z[:,1]).max():.3e}')
print(f'  Max |delta_S|  = {np.abs(diff_SI[:,0]).max():.3e}')
print(f'  Max |delta_I|  = {np.abs(diff_SI[:,1]).max():.3e}')
print(f'  RMSE z1        = {np.sqrt(np.mean(diff_z[:,0]**2)):.3e}')
print(f'  RMSE z2        = {np.sqrt(np.mean(diff_z[:,1]**2)):.3e}')

PASS = np.abs(diff_z).max() < 1e-6
print()
print('PASS  (max diff < 1e-6 -- implementations agree)' if PASS
      else 'FAIL  (diff > 1e-6 -- check sigma-point or weight mismatch)')

# ---- overlay plot -------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tt = np.arange(T_fp)

for ax, j, lbl in zip(axes, [0, 1], ['S (susceptible)', 'I (infectious)']):
    ax.plot(tt, fp_filt_SI[:, j],  color='steelblue', lw=2,
            label='filterpy' if _USE_FILTERPY else 'numpy mirror')
    ax.plot(tt, hc_filt_SI[:, j],  color='crimson',   lw=1.5, ls='--',
            label='hand-coded')
    ax.plot(tt, true_SI[:, j],     color='black',      lw=1,   ls=':',
            label='true', alpha=0.7)
    ax.set_title(lbl);  ax.set_xlabel('week')
    ax.legend(fontsize=9);  ax.grid(True, alpha=0.3)

src_lbl = 'filterpy library' if _USE_FILTERPY else 'numpy mirror of filterpy'
fig.suptitle(f'filterpy ({src_lbl}) vs hand-coded UKF', fontsize=12)
fig.tight_layout()
savefig(fig, 'filterpy_comparison')
plt.show()


## Test 1 — Parameter Recovery (Monte Carlo)
### 1a: Single-dataset estimation

In [ ]:
# ── Cell F: True parameters + simulation helper ──────────────────────────────
# β=0.3, γ=0.1, R0=3 → epidemic peaks ~t30-40 and decays by t80 within T=100

true_params = {
    'beta': 0.3,   'gamma': 0.1,
    'I0':   0.01,  'S0':    0.99,
    'c2':   200.0, 'c3':   -0.8,
    'a1':   0.0,   'a2':    50.0, 'a3': -0.5,
    'sigma2_S':    0.005,
    'sigma2_I':    0.15,
    'sigma2_eps1': 0.005,
    'sigma2_eps2': 50.0,
    'sigma2_eps3': 0.1,
}

def simulate_sir_obs(params, T=100, dt=1.0, rng=None, min_peak_I=1e-3):
    """
    Simulate T weekly observations from the ALR-space SIR model.

    Returns (Y, M, true_SI, true_Z)  or None if epidemic is degenerate.
      Y       : (T, 3) observations  [GT, Articles, Tone]
      M       : (T, 3) all-True mask
      true_SI : (T, 2) true [S, I]
      true_Z  : (T, 2) true ALR z-states
    """
    if rng is None:
        rng = np.random.default_rng()

    beta  = params['beta'];   gamma = params['gamma']
    I0    = params['I0'];     S0    = params['S0']
    c2    = params['c2'];     c3    = params['c3']
    a1    = params.get('a1', 0.0)
    a2    = params['a2'];     a3    = params['a3']
    s2_S  = params['sigma2_S'];    s2_I  = params['sigma2_I']
    s2_e1 = params['sigma2_eps1']; s2_e2 = params['sigma2_eps2']
    s2_e3 = params['sigma2_eps3']

    sq_S  = np.sqrt(s2_S);  sq_I = np.sqrt(s2_I)
    sq_e1 = np.sqrt(s2_e1); sq_e2 = np.sqrt(s2_e2); sq_e3 = np.sqrt(s2_e3)

    z       = _alr(S0, I0).copy()
    true_Z  = np.zeros((T, 2))
    true_SI = np.zeros((T, 2))
    Y       = np.zeros((T, 3))

    for t in range(T):
        SI      = _inv_alr_rows(z[None, :])[0]
        SI_next = sir_transition_vec(SI[None, :], beta, gamma, dt)[0]
        z_det   = _alr_rows(SI_next[None, :])[0]
        # process noise in ALR (z) space
        z       = z_det + np.array([sq_S  * rng.standard_normal(),
                                     sq_I  * rng.standard_normal()])
        SI_t    = _inv_alr_rows(z[None, :])[0]
        true_Z[t]  = z
        true_SI[t] = SI_t
        I_t = SI_t[1]

        Y[t, 0] = 1.0 * I_t + a1 + sq_e1 * rng.standard_normal()
        Y[t, 1] = c2  * I_t + a2 + sq_e2 * rng.standard_normal()
        Y[t, 2] = c3  * I_t + a3 + sq_e3 * rng.standard_normal()

    if true_SI[:, 1].max() < min_peak_I:
        return None

    M = np.ones((T, 3), dtype=bool)
    return Y, M, true_SI, true_Z

print('true_params and simulate_sir_obs defined.')
print(f"R0 = beta/gamma = {true_params['beta']/true_params['gamma']:.1f}")

In [ ]:
# ── Cell E: negloglik_logit, _run_filter_logit, estimate_logit ───────────────

def negloglik_logit(params, obs, masks, names, channels, dt,
                    art_scale=1.0, tone_scale=1.0):
    """ALR-space UKF negative log-likelihood (scalar)."""
    try:
        p   = _theta_to_p(params, art_scale, tone_scale)
        nll = _ukf_pass(p, obs, masks, dt=dt, return_all=False)
        return float(nll) if np.isfinite(nll) else 1e12
    except Exception:
        return 1e12


def _run_filter_logit(p, obs, masks, channels, dt=1.0, return_P=False):
    """
    Run ALR-space UKF and return filtered states + standardised residuals.

    Parameters
    ----------
    p        : natural-space parameter dict
    obs      : (T, C) observations
    masks    : (T, C) bool array
    channels : list of channel names (unused here, kept for API compat)
    dt       : time step
    return_P : if True also return (P_list, filt_z)

    Returns
    -------
    filt_SI   : (T, 2) filtered [S, I]
    std_res   : (T, C) standardised innovations
    [P_list   : list of T (2,2) filtered covariances]
    [filt_z   : (T, 2) filtered ALR states]
    """
    _, filt_SI, std_res, P_list, filt_z = _ukf_pass(
        p, obs, masks, dt=dt, return_all=True)
    if return_P:
        return filt_SI, std_res, P_list, filt_z
    return filt_SI, std_res


def estimate_logit(obs, masks, config, dt=1.0,
                   beta0=0.1, gamma0=0.05,
                   art_scale=1.0, tone_scale=1.0,
                   q_lo=-10.0, q_hi=5.0,
                   n_restarts=5, verbose=False):
    """
    MLE for ALR-space UKF via multi-start L-BFGS-B.

    Returns
    -------
    (param_dict, opt_result)
        param_dict : natural-space parameters (includes derived R0)
        opt_result : scipy OptimizeResult from best restart
    """
    names = config['names']

    def obj(theta):
        return negloglik_logit(theta, obs, masks, names,
                               config['channels'], dt,
                               art_scale=art_scale, tone_scale=tone_scale)

    # Bounds for 12-element optimization vector
    bounds = [
        (-6.0,  1.0),    # log_beta
        (-6.0,  1.0),    # log_gamma
        (-10.0, -0.7),   # log_I0   (I0 in [~4e-5, 0.49])
        (-4.0,  8.0),    # log_c2n
        (-5.0,  5.0),    # c3
        (-5.0,  5.0),    # a2n
        (-5.0,  5.0),    # a3n
        (q_lo,  q_hi),   # log_s2S
        (q_lo,  q_hi),   # log_s2I
        (q_lo,  q_hi),   # log_s2e1
        (q_lo,  q_hi),   # log_s2e2n
        (q_lo,  q_hi),   # log_s2e3
    ]

    rng_opt = np.random.default_rng(seed=0)
    best    = None

    for restart in range(n_restarts):
        if restart == 0:
            # Informed starting point
            I0_guess = np.clip(float(np.nanmean(obs[:, 0])) / max(art_scale, 1.0),
                                1e-5, 0.1)
            theta0 = np.array([
                np.log(beta0), np.log(gamma0), np.log(I0_guess),
                0.0,  -0.5,          # log_c2n, c3
                0.0,  -0.5,          # a2n, a3n
                np.log(0.01), np.log(0.1),          # log_s2S, log_s2I
                np.log(0.01), np.log(1.0), np.log(0.1),  # noise vars
            ])
        else:
            lo     = np.array([b[0] for b in bounds])
            hi     = np.array([b[1] for b in bounds])
            theta0 = rng_opt.uniform(lo, hi)

        try:
            res = minimize(obj, theta0, method='L-BFGS-B', bounds=bounds,
                           options={'maxiter': 3000, 'ftol': 1e-11, 'gtol': 1e-7})
            if verbose:
                print(f'  restart {restart+1:2d}/{n_restarts}: '
                      f'nll={res.fun:.4f}  success={res.success}')
            if best is None or res.fun < best.fun:
                best = res
        except Exception as e:
            if verbose:
                print(f'  restart {restart+1} failed: {e}')

    p_est       = _theta_to_p(best.x, art_scale, tone_scale)
    p_est['R0'] = p_est['beta'] / p_est['gamma']
    return p_est, best

print('negloglik_logit, _run_filter_logit, estimate_logit defined.')

In [ ]:
# ── Cell D: CONFIGS, parameter pack/unpack, UKF pass ─────────────────────────

CONFIGS_LOGIT = {
    '2int': {
        # 3 channels; c1=1 fixed, a1=0 fixed (no GT intercept)
        'channels': ['gt', 'art', 'tone'],
        # 12-element optimization vector (log/normalised)
        'names': ['log_beta', 'log_gamma', 'log_I0',
                  'log_c2n',  'c3',
                  'a2n',      'a3n',
                  'log_s2S',  'log_s2I',
                  'log_s2e1', 'log_s2e2n', 'log_s2e3'],
        'n_params': 12,
    }
}

# ── parameter conversions ──────────────────────────────────────────────────

def _theta_to_p(theta, art_scale=1.0, tone_scale=1.0):
    """Optimization vector → natural-space parameter dict."""
    (log_b, log_g, log_I0,
     log_c2n, c3,
     a2n, a3n,
     log_s2S, log_s2I,
     log_s2e1, log_s2e2n, log_s2e3) = theta
    I0 = float(np.clip(np.exp(log_I0), 1e-8, 0.49))
    return {
        'beta':        float(np.exp(log_b)),
        'gamma':       float(np.exp(log_g)),
        'I0':          I0,
        'S0':          1.0 - I0,
        'c2':          float(np.exp(log_c2n) * art_scale),
        'c3':          float(c3),
        'a1':          0.0,
        'a2':          float(a2n * art_scale),
        'a3':          float(a3n * tone_scale),
        'sigma2_S':    float(np.exp(log_s2S)),
        'sigma2_I':    float(np.exp(log_s2I)),
        'sigma2_eps1': float(np.exp(log_s2e1)),
        'sigma2_eps2': float(np.exp(log_s2e2n) * art_scale**2),
        'sigma2_eps3': float(np.exp(log_s2e3)),
    }

def _p_to_theta(p, art_scale=1.0, tone_scale=1.0):
    """Natural-space param dict → optimization vector."""
    eps = 1e-12
    return np.array([
        np.log(max(p['beta'],   eps)),
        np.log(max(p['gamma'],  eps)),
        np.log(max(p['I0'],     eps)),
        np.log(max(p['c2'] / max(art_scale,  eps), eps)),
        p['c3'],
        p['a2'] / max(art_scale,  eps),
        p['a3'] / max(tone_scale, eps),
        np.log(max(p['sigma2_S'],    eps)),
        np.log(max(p['sigma2_I'],    eps)),
        np.log(max(p['sigma2_eps1'], eps)),
        np.log(max(p['sigma2_eps2'] / max(art_scale**2, eps), eps)),
        np.log(max(p['sigma2_eps3'], eps)),
    ])

# ── core UKF forward pass ─────────────────────────────────────────────────

def _ukf_pass(p, obs, masks, dt=1.0, return_all=False):
    """
    Full UKF forward pass in ALR space.

    Returns
    -------
    return_all=False : float  (negative log-likelihood)
    return_all=True  : (nll, filt_SI, std_resids, P_list, filt_z)
    """
    T, C   = obs.shape
    beta   = p['beta'];  gamma = p['gamma']
    c_sc   = np.array([1.0,  p['c2'],  p['c3']])
    intcpt = np.array([p.get('a1', 0.0), p['a2'], p['a3']])
    Q      = np.diag([p['sigma2_S'], p['sigma2_I']])
    Rn     = np.diag([p['sigma2_eps1'], p['sigma2_eps2'], p['sigma2_eps3']])

    mu = _alr(p['S0'], p['I0'])
    P  = 10.0 * Q + 1e-6 * np.eye(2)

    nll = 0.0
    if return_all:
        filt_z  = np.zeros((T, 2))
        filt_SI = np.zeros((T, 2))
        std_res = np.full((T, C), np.nan)
        P_list  = []

    for t in range(T):
        # ── Predict ──────────────────────────────────────────────────────
        sp      = _sigma_points(mu, P)               # (5,2) ALR
        SI_sp   = _inv_alr_rows(sp)                   # (5,2)
        SI_pr   = sir_transition_vec(SI_sp, beta, gamma, dt)
        z_pr    = _alr_rows(SI_pr)                    # (5,2)

        mu_pr   = _WM @ z_pr
        dz      = z_pr - mu_pr
        P_pr    = dz.T @ _CW @ dz + Q
        P_pr    = 0.5 * (P_pr + P_pr.T) + 1e-10 * np.eye(2)

        # ── Update ───────────────────────────────────────────────────────
        mask_t  = masks[t]
        obs_t   = obs[t]

        sp2     = _sigma_points(mu_pr, P_pr)
        SI_m    = _inv_alr_rows(sp2)
        Y_sp    = c_sc * SI_m[:, 1:2] + intcpt        # (5,3)
        y_pr    = _WM @ Y_sp                           # (3,)

        if mask_t.any():
            idx       = np.where(mask_t)[0]
            y_obs     = obs_t[idx]
            Yobs      = Y_sp[:, idx];  y_pr_o = y_pr[idx]
            R_o       = Rn[np.ix_(idx, idx)]

            dy        = Yobs - y_pr_o
            Syy       = dy.T @ _CW @ dy + R_o
            Syy       = 0.5 * (Syy + Syy.T)

            dz2       = sp2 - mu_pr
            Pzy       = dz2.T @ _CW @ dy               # (2, n_o)

            try:
                Syyinv = np.linalg.inv(Syy)
            except np.linalg.LinAlgError:
                Syyinv = np.linalg.pinv(Syy)

            K      = Pzy @ Syyinv
            innov  = y_obs - y_pr_o

            mu = mu_pr + K @ innov
            P  = P_pr  - K @ Syy @ K.T
            P  = 0.5 * (P + P.T) + 1e-10 * np.eye(2)

            sgn, ldet = np.linalg.slogdet(Syy)
            if sgn > 0:
                n_o  = len(idx)
                nll -= 0.5 * (n_o * np.log(2 * np.pi) + ldet
                               + innov @ Syyinv @ innov)
            else:
                nll += 1e6

            if return_all:
                n_o = len(idx)
                try:
                    Ls  = np.linalg.cholesky(Syy + 1e-12 * np.eye(n_o))
                    sr  = np.linalg.solve(Ls, innov)
                    for j, i in enumerate(idx):
                        std_res[t, i] = sr[j]
                except Exception:
                    for j, i in enumerate(idx):
                        std_res[t, i] = innov[j] / max(np.sqrt(Syy[j, j]), 1e-12)
        else:
            mu = mu_pr
            P  = P_pr

        if return_all:
            filt_z[t]  = mu
            filt_SI[t] = _inv_alr_rows(mu[None, :])[0]
            P_list.append(P.copy())

    neg_nll = -nll
    if return_all:
        return neg_nll, filt_SI, std_res, P_list, filt_z
    return neg_nll

print('CONFIGS_LOGIT and UKF pass defined.')

In [ ]:
# ── Cell C: UKF weights (precomputed globals) ────────────────────────────────

_N_STATE  = 2
_ALPHA    = 1e-3
_BETA_UKF = 2.0
_KAPPA    = 0.0
_LAMBDA   = _ALPHA**2 * (_N_STATE + _KAPPA) - _N_STATE
_N_SP     = 2 * _N_STATE + 1   # 5 sigma points

_WM       = np.empty(_N_SP)
_WC       = np.empty(_N_SP)
_WM[0]    = _LAMBDA / (_N_STATE + _LAMBDA)
_WC[0]    = _WM[0] + (1.0 - _ALPHA**2 + _BETA_UKF)
_WM[1:]   = _WC[1:] = 0.5 / (_N_STATE + _LAMBDA)
_CW       = np.diag(_WC)       # (5,5) weight matrix for cov computation

def _sigma_points(mu, P):
    """Generate 2n+1 sigma points from mean mu and covariance P."""
    n     = len(mu)
    scale = np.sqrt(n + _LAMBDA)
    Ps    = 0.5 * (P + P.T) + 1e-10 * np.eye(n)
    try:
        L = np.linalg.cholesky(Ps)
    except np.linalg.LinAlgError:
        L = np.diag(np.sqrt(np.maximum(np.diag(Ps), 1e-12)))
    pts       = np.empty((2*n+1, n))
    pts[0]    = mu
    for i in range(n):
        pts[i+1]   = mu + scale * L[:, i]
        pts[n+i+1] = mu - scale * L[:, i]
    return pts

print(f'UKF weights: WM[0]={_WM[0]:.6f}, WC[0]={_WC[0]:.6f}, WM[1]={_WM[1]:.6f}')

In [ ]:
# ── Cell B: ALR transforms + SIR transition ──────────────────────────────────

def _alr(S, I):
    """Maps (S, I) scalars → 2-vector in ALR z-space."""
    R = max(1.0 - float(S) - float(I), 1e-12)
    return np.array([np.log(float(S) / R), np.log(float(I) / R)])

def _inv_alr_rows(Z):
    """Maps (k, 2) ALR matrix → (k, 2) [S, I] on the simplex."""
    Z = np.asarray(Z, dtype=float)
    e = np.exp(np.clip(Z, -50.0, 50.0))
    d = 1.0 + e[:, 0] + e[:, 1]
    return np.column_stack([e[:, 0] / d, e[:, 1] / d])

def _alr_rows(SI):
    """Maps (k, 2) [S, I] matrix → (k, 2) z-space."""
    SI = np.asarray(SI, dtype=float)
    S, I = SI[:, 0], SI[:, 1]
    R = np.maximum(1.0 - S - I, 1e-12)
    return np.column_stack([np.log(S / R), np.log(I / R)])

def sir_transition_vec(X, beta, gamma, dt):
    """Euler SIR step. X: (k, 2) of [S, I]. Returns (k, 2)."""
    X = np.asarray(X, dtype=float)
    S, I = X[:, 0], X[:, 1]
    inf   = beta * S * I * dt
    S_new = np.clip(S - inf,                   1e-9, None)
    I_new = np.clip(I + inf - gamma * I * dt,  1e-9, None)
    over  = (S_new + I_new) > 1.0
    if over.any():
        tot = S_new[over] + I_new[over]
        S_new[over] /= tot;  I_new[over] /= tot
    return np.column_stack([S_new, I_new])

print('ALR + SIR functions defined.')

In [ ]:
# ── Cell A: Imports & setup ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, warnings
from scipy.optimize import minimize
from scipy.stats import chi2, jarque_bera as jb_test
from statsmodels.stats.diagnostic import acorr_ljungbox

warnings.filterwarnings('ignore')
np.random.seed(42)

os.makedirs('figures_logit', exist_ok=True)
os.makedirs('cache_logit',  exist_ok=True)

def savefig(fig, name):
    fig.savefig(f'figures_logit/{name}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved figures_logit/{name}.png')

print('Setup complete.')

# Simulation-Based Validation for ALR-Space UKF (SIR Epidemic Model)

This section implements and validates a full Unscented Kalman Filter pipeline for an SIR epidemic model in additive log-ratio (ALR) coordinates with three observables: Google Trends, GDELT articles, and GDELT tone.